# Suite SYN — Syntax trees

Every language's trees implement the same protocols (`mbse.Programs.Framework.Syntax`): kinds declared once as typed
properties, syntax nodes that serialize through derived meta-schemas and builders, validation, standards, and traversal by
properties alone. This suite checks that machinery on a small language declared here, Toy, and on Ccpp where real trees
help.

In [ ]:
from __future__ import annotations

from typing import Literal as Choice

import mbse.Programs
from mbse.Programs.Ccpp import Syntax as S
from mbse.Programs.Framework import Syntax as F
from mbse.Programs.Framework.Syntax import Attribute, Child, Language, SyntaxNode
from mbse.Programs.TypeScript import Syntax as TS
from mbse.Schemas.Framework import JSON, Plain, Proxies, Visitors
from support import raises


class Toy(SyntaxNode):
    """A category of Toy."""


class Leaf(Toy):
    value: str
    flag: bool
    count: int | None
    color: Choice["red", "blue"] | None
    SINCE = {"Toy": 2000}
    FEATURES = {"color": {"blue": {"Toy": 2010}}, "flag": {True: {"Toy": (2000, 2020)}}}


class Pair(Toy):
    left: Toy
    right: Toy | None
    items: list[Toy]

    def features(self):
        return [("Pair with three items", {"Toy": 2030})] if len(self.items) == 3 else []

    def check(self):
        return ["a Pair cannot hold itself"] if self.left is self else []


class Extra(Toy):
    leaf: Leaf | None
    EXTENSION = True


TOY = Language("Toy", [Leaf, Pair, Extra], base={"Toy": 1990})


def leaf(value="a", **fields):
    return Leaf(value=value, **fields)

## SYN-01 · Kinds declare their properties as typed annotations

In [ ]:
assert [f.name for f in Leaf.PROPERTIES] == ["value", "flag", "count", "color"]
assert Leaf.PROPERTIES[0] == Attribute("value", str) and Leaf.PROPERTIES[1] == Attribute("flag", bool)
assert Leaf.PROPERTIES[2] == Attribute("count", int, optional=True)
assert Leaf.PROPERTIES[3] == Attribute("color", str, optional=True, choices=("red", "blue"))
assert Pair.PROPERTIES == (Child("left", (Toy,)), Child("right", (Toy,), optional=True), Child("items", (Toy,), True, True))
assert Leaf.PROPERTIES[3].describe() == {"name": "color", "type": "str", "optional": True, "choices": ["red", "blue"]}
assert Pair.PROPERTIES[2].describe() == {"name": "items", "type": ["Toy"], "optional": True, "many": True}
assert Leaf.KIND == "Leaf" and Leaf.NAME == "Programs.Toy.Leaf" and Leaf.LANGUAGE is TOY
assert S.IfStatement.PROPERTIES[4] == Child("condition", (S.Expression, S.Declaration), optional=True)


class Bad(Toy):
    numbers: list[int]


with raises(TypeError, match="Bad.numbers: a property holds a native, a choice, or syntax nodes"):
    Language("Bad", [Bad], base={})


class Lazy(Toy):  # this cell does not import annotations from __future__: Python 3.14 evaluates them lazily
    leaf: Leaf | None


Language("Lazy", [Lazy], base={})
assert Lazy.PROPERTIES == (Child("leaf", (Leaf,), optional=True),)
print("ok")

## SYN-02 · Syntax nodes are plain objects: construction, defaults, `repr`, identity

In [ ]:
made = leaf("x", flag=True, count=3)
assert (made.value, made.flag, made.count, made.color) == ("x", True, 3, None)
pair = Pair(left=made, items=(leaf("y"),))
assert pair.right is None and isinstance(pair.items, list) and Pair(left=made).items == []
assert repr(pair) == "Pair(left=Leaf(value='x', flag=True, count=3), items=[Leaf(value='y')])"
assert repr(leaf("z", count=0)) == "Leaf(value='z', count=0)"
assert Leaf().flag is False and Leaf().value is None
with raises(TypeError, match="Toy is not a kind of syntax node"):
    Toy()
with raises(TypeError, match="Leaf has no property 'colour'"):
    Leaf(colour="red")
assert made.identity() == id(made) and made.schema_name() == "Programs.Toy.Leaf" and made.owner() is None
assert made.check() == [] and made.features() == [] and made.availability() == {"Toy": 2000}
assert F.children(Pair(left=made, items=[None, made])) == [("left", None, made), ("items", 1, made)]
emptied = Pair(left=made)
emptied.items = None
assert F.children(emptied) == [("left", None, made)] and mbse.Programs.Ccpp.Syntax is S
print("ok")

## SYN-03 · Meta-schemas, registration and snapshots

In [ ]:
assert TOY.Builders.schema("Programs.Toy.Leaf") is Leaf.Schema
assert list(Leaf.Schema.properties) == ["kind", "value", "flag", "count", "color"]
assert list(Leaf.Schema.adjacencies) == ["parent"] and list(Pair.Schema.adjacencies) == ["children", "parent"]
assert [b.name for b in TOY.Schema.branches] == ["Leaf", "Pair", "Extra"]
tree = Pair(left=leaf("x", flag=True), right=leaf("y", color="blue", count=0), items=[leaf("z"), leaf("w")])
plain = Plain.ToPlain(TOY.Builders).Reachable(Pair.Schema, tree)
assert plain["objects"]["s0"] == {"kind": "Pair", "children": [
    {"child": {"$ref": "s1", "$schema": "Programs.Toy.Leaf"}, "property": "left"},
    {"child": {"$ref": "s2", "$schema": "Programs.Toy.Leaf"}, "property": "right"},
    {"child": {"$ref": "s3", "$schema": "Programs.Toy.Leaf"}, "property": "items", "index": 0},
    {"child": {"$ref": "s4", "$schema": "Programs.Toy.Leaf"}, "property": "items", "index": 1}]}
assert plain["objects"]["s1"] == {"kind": "Leaf", "value": "x", "flag": True}  # False is not written
assert plain["objects"]["s2"] == {"kind": "Leaf", "value": "y", "count": 0, "color": "blue"}
copy = JSON.FromJSON(TOY.Builders).Reachable(Pair.Schema, JSON.ToJSON(TOY.Builders).Reachable(Pair.Schema, tree))
assert F.same(copy, tree) and copy is not tree and copy.left.flag is True and copy.right.flag is False
# The language's store holds the nodes its builders make, decoding's included; a builder it did not make adds none.
assert any(node is copy for node in TOY.Builders.extent("Programs.Toy.Pair")) and TOY.Builders.extent("Programs.Toy.Extra") == ()
direct = type(TOY.Builders.Leaf())().set("value", "d").create()
assert all(node is not direct for node in TOY.Builders.extent("Programs.Toy.Leaf"))
with raises(AttributeError, match="no schema registered as 'Programs.Toy.Nope'"):
    TOY.Builders.builder("Programs.Toy.Nope")
unit = S.TranslationUnit(items=[S.ExpressionStatement(expression=S.IntegerLiteral(spelling="1"))])
assert F.same(JSON.FromJSON(S.LANGUAGE.Builders).Reachable(S.TranslationUnit.Schema,
                                                         JSON.ToJSON(S.LANGUAGE.Builders).Reachable(S.TranslationUnit.Schema, unit)), unit)
print("ok")

## SYN-04 · Builders: the DSL, fluent setters and their specs, `create`/`clone`/`update`, and their errors

In [ ]:
builder = TOY.builder("Pair").set("left", leaf("x")).add("items", leaf("y")).add("items", leaf("z"))
made = builder.create()
assert [i.value for i in made.items] == ["y", "z"] and made.left.value == "x"
again = TOY.builder(Pair, made).set("items", [leaf("q")]).set("right", leaf("r")).clone()
assert again is not made and [i.value for i in again.items] == ["q"] and made.right is None
updated = TOY.builder(Pair, made).set("right", None).set("left", leaf("l")).update()
assert updated is made and made.left.value == "l"
assert TOY.builder("Leaf").set("value", "v").set("flag", True).create().flag is True
with raises(KeyError, match="Toy has no kind 'Tree'"):
    TOY.builder("Tree")
with raises(TypeError, match="expected a Leaf to build from, got Pair"):
    TOY.builder(Leaf, made)
with raises(KeyError, match="Pair has no child property 'middle'"):
    TOY.builder(Pair).set("middle", leaf())
with raises(TypeError, match="Pair.left holds one child; use set()"):
    TOY.builder(Pair).add("left", leaf())
with raises(ValueError, match="create() is only valid without a source instance"):
    TOY.builder(Pair, made).create()
with raises(ValueError, match="clone() is only valid with a source instance"):
    TOY.builder(Pair).clone()
with raises(ValueError, match="update() is only valid with a source instance"):
    TOY.builder(Pair).update()

# Fluent setters, one per property: a child's takes a syntax node, a builder, or a function that makes one, from the builder of
# the property's kind if it holds one kind, and from the language's builders otherwise
Toys = TOY.Builders
fluent = Toys.Pair().left(leaf("x")).items([Toys.Leaf().value("y"), leaf("z")]).add_items(
    lambda b: b.Leaf().value("w")).create()
assert fluent.left.value == "x" and [i.value for i in fluent.items] == ["y", "z", "w"]
assert Toys.Pair().left(lambda b: b.Leaf().value("n").flag(True)).create().left.flag is True
assert Toys.Extra().leaf(lambda b: b.value("v")).create().leaf.value == "v"
assert Toys.Extra().leaf(lambda b: leaf("m")).create().leaf.value == "m"
cloned = Toys.Pair(fluent).left(Toys.Leaf(fluent.left).value("x2")).right(leaf()).right(None).clone()
assert cloned.left.value == "x2" and cloned.right is None and fluent.left.value == "x"
assert S.LANGUAGE.Builders.GotoStatement().label("done").create().label.spelling == "done"  # an Identifier's spelling
assert TS.LANGUAGE.Builders.ForStatement().update_(lambda b: b.Identifier().name("i")).create().update.name == "i"
with raises(TypeError, match="Pair.left takes a syntax node, a builder or a function that makes one, got str"):
    Toys.Pair().left("x")
with raises(TypeError, match="the function for Pair.left must return a builder or a syntax node, got str"):
    Toys.Pair().left(lambda b: "x")


def built(*entries):
    """A Pair built from children entries given as (link target, property, index), through the visitor protocol."""
    b = TOY.builder(Pair)

    def fill(target, prop, index):
        def write(entry):
            if target is not None:
                entry.link("child", lambda k: k.set(target))
            if prop is not None:
                entry.property("property", lambda p: p.value(lambda a: a.as_native(lambda n: n.set(prop))))
            if index is not None:
                entry.property("index", lambda p: p.value(lambda a: a.as_native(lambda n: n.set(index))))
        return write

    for e in entries:
        b.adjacency("children", lambda a, e=e: a.add(fill(*e)))
    return b.create()


x, y, z = leaf("x"), leaf("y"), leaf("z")
made = built((z, "items", None), (y, "items", 1), (x, "items", 0), (x, "left", None))
assert [i.value for i in made.items] == ["x", "y", "z"]  # by index; entries without one last
assert [i.value for i in built((z, "items", None), (y, "items", None), (x, "left", None)).items] == ["z", "y"]
with raises(ValueError, match="link 'child' is not set"):
    built((None, "left", None))
with raises(ValueError, match="a child entry needs a property"):
    built((x, None, None))
with raises(ValueError, match="a Pair has no child property 'middle'"):
    built((x, "middle", None))
with raises(TypeError, match="a child must be a syntax node, got str"):
    built(("x", "left", None))
with raises(ValueError, match="Pair.left holds one child, not a list"):
    built((x, "left", 0))
with raises(ValueError, match="Pair.left holds one child, got several"):
    built((x, "left", None), (y, "left", None))
print("ok")

## SYN-05 · Builders, entries and links implement the visitor protocols

In [ ]:
b = TOY.builder(Leaf, leaf("v", count=2))
names = []
b.properties(lambda p: names.append((p.name(), p.has())))
assert names == [("kind", True), ("value", True), ("count", True)]
assert b.has("kind") and not b.has("flag") and not b.has("nothing")
b.property("flag", lambda p: p.value(lambda a: a.as_native(lambda n: n.set(True))))
assert b.has("flag")
read = []
b.property("value", lambda p: p.value(lambda a: a.as_native(lambda n: read.append(n.get()))))
assert read == ["v"]
b.property("kind", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("Leaf"))))
with raises(ValueError, match="expected kind 'Leaf', got 'Pair'"):
    b.property("kind", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("Pair"))))
with raises(TypeError, match="expected int, got str"):
    b.property("count", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("2"))))
with raises(KeyError, match="unknown property 'size'"):
    b.property("size", lambda p: None)
b.clear("count").clear("kind").clear("flag").clear("nothing")
assert not b.has("count") and b.has("kind") and not b.has("flag")
with raises(AttributeError, match="property 'count' is not set"):
    b.property("count", lambda p: p.value(lambda a: a.as_native(lambda n: n.get())))
for kind in ("as_object", "as_union", "as_intersection", "as_indexed"):
    with raises(TypeError, match="property 'value' is native"):
        b.property("value", lambda p, kind=kind: p.value(lambda a: getattr(a, kind)(lambda v: None)))
assert b.identify(made) is b
seen = []
b.adjacencies(lambda a: seen.append((a.name(), a.me())))
assert seen == [("parent", "child")]
with raises(KeyError, match="unknown adjacency 'children'"):
    b.adjacency("children", lambda a: None)
b.adjacency("parent", lambda a: a.add(lambda e: None).entries(lambda e: seen.append(e)).remove(None))
assert len(seen) == 1  # `parent` holds no entries: parents' children imply them

p = TOY.builder(Pair, Pair(left=x, items=[y]))
seen = []
p.adjacencies(lambda a: seen.append(a.name()))
assert seen == ["children", "parent"]
entries = []
p.adjacency("children", lambda a: a.entries(entries.append))
first = entries[0]
links, fields = [], []
first.links(lambda k: links.append(k.name()))
first.link("child", lambda k: k.target(lambda t: links.append(t)))
first.properties(lambda q: fields.append((q.name(), q.has())))
assert links == ["child", x] and fields == [("property", True)]
indexed = []
entries[1].properties(lambda q: indexed.append((q.name(), q.has())))
assert indexed == [("property", True), ("index", True)]
entries[1].clear("index")
assert not entries[1].has("index")
assert first.has("property") and not first.has("index") and not first.has("other")
with raises(KeyError, match="'parent' is not a link this entry can set"):
    first.link("parent", lambda k: None)
with raises(KeyError, match="unknown property 'other'"):
    first.property("other", lambda q: None)
first.clear("property").clear("other")
assert not first.has("property")
p.adjacency("children", lambda a: a.remove(entries[1]))
with raises(ValueError, match="a child entry needs a property"):
    p.clone()  # the first entry lost its property
first.property("property", lambda q: q.value(lambda a: a.as_native(lambda n: n.set("left"))))
assert p.clone().items == []  # and the second was removed
with raises(ValueError, match="link 'child' is not set"):
    p.adjacency("children", lambda a: a.add(lambda e: e.links(lambda k: k.target(lambda t: None))))
print("ok")

## SYN-06 · The registry builds syntax nodes from snapshots

In [ ]:
registry = TOY.Builders
assert registry.schema("Programs.Toy.Leaf") is Leaf.Schema and registry.name_of(Pair.Schema) == "Programs.Toy.Pair"
assert registry.member(leaf("m"), "value") == "m"
with raises(TypeError, match="'Programs.Children' is a relation; no relation builder is exposed"):
    registry.schema(F.CHILDREN)
with raises(AttributeError, match="no schema registered as 'Programs.Toy.Tree'"):
    registry.schema("Programs.Toy.Tree")
with raises(LookupError, match="schema is not registered"):
    registry.name_of(S.Identifier.Schema)
assert TOY.name() == "Toy" and repr(TOY) == "<language Toy>"
assert list(TOY.kinds()) == ["Leaf", "Pair", "Extra"] and list(TOY.categories()) == ["Toy"]
assert TOY.schema_of(leaf()) is Leaf.Schema
with raises(TypeError, match="not a Toy syntax node: 3"):
    TOY.schema_of(3)
grammar = TOY.grammar()
assert grammar["language"] == "Toy" and grammar["base"] == {"Toy": 1990}
assert grammar["categories"] == [{"name": "Toy", "base": "SyntaxNode"}]
assert grammar["kinds"][0] == {"kind": "Leaf", "category": "Toy", "since": {"Toy": 2000}, "extension": False,
                               "features": {"color": {"blue": {"Toy": 2010}}, "flag": {"true": {"Toy": [2000, 2020]}}},
                               "properties": [f.describe() for f in Leaf.PROPERTIES]}
assert grammar["kinds"][1]["since"] is None and grammar["kinds"][2]["extension"] is True
print("ok")

## SYN-07 · Validation: required properties, categories, choices, sharing, cycles and kinds' own checks

In [ ]:
assert TOY.validate(Pair(left=leaf(), items=[leaf()])) == []
assert TOY.validate(Pair()) == ["a Pair needs a left"]
assert TOY.validate(Leaf(count="3", color="green")) == [
    "a Leaf needs a value", "Leaf.count must be an int, got str", "Leaf.color cannot be 'green'"]
assert TOY.validate(Leaf(value=3, flag=1)) == ["Leaf.value must be a str, got int", "Leaf.flag must be a bool, got int"]
shared = leaf()
assert TOY.validate(Pair(left=shared, items=[leaf(), shared])) == ["items[1]: the syntax node is also at left"]
cycle = Pair(items=[])
cycle.left = cycle
assert TOY.validate(cycle) == ["left: the syntax node is also at the root", "a Pair cannot hold itself"]
assert TOY.validate(Pair(left=S.Identifier(spelling="a"), right="b", items=[None, 3])) == [
    "left: Pair.left must be a Toy, got an Identifier", "right: Pair.right must be a Toy, got str",
    "Pair.items[0] is empty", "items[1]: Pair.items must be a Toy, got int"]
not_a_list = Pair(left=leaf())
not_a_list.items = leaf()
assert TOY.validate(not_a_list) == ["Pair.items must be a list, got Leaf"]
assert TOY.validate(Pair(left=Pair(left=Leaf()))) == ["left.left: a Leaf needs a value"]
assert TOY.validate("x") == ["not a Toy syntax node: 'x'"]
assert S.LANGUAGE.validate(S.BinaryExpression(left=S.IdExpression(name=S.Identifier(spelling="9")), operator="+",
                                              right=S.IntegerLiteral(spelling="1"))) == [
    "left.name: '9' is not an identifier"]
print("ok")

## SYN-08 · Standards: availability of kinds and features, and their problems

In [ ]:
toy2015, toy2025 = F.Standard(TOY, "Toy", 2015), F.Standard(TOY, "Toy", 2025)
assert toy2015.name() == "Toy15" and repr(toy2025) == "<standard Toy25>" and F.label("C++", 1998) == "C++98"
assert toy2015.version == 2015 and toy2015.label(1998) == "Toy98"


class Numbered(F.Standard):
    def label(self, version):
        return f"Toy {version}"


assert Numbered(TOY, "Toy", 1995).problems(leaf()) == ["Leaf needs Toy 2000"]
assert Numbered(TOY, "Toy", 2025).name() == "Toy 2025"
assert toy2015.has({"Toy": 2000}) and not toy2015.has({"Toy": 2020}) and not toy2015.has({"Other": 1})
assert toy2015.has({"Toy": (2000, 2020)}) and not toy2025.has({"Toy": (2000, 2020)})
assert toy2015.problems(leaf(color="blue", flag=True)) == []
assert toy2025.problems(leaf(color="blue", flag=True)) == ["Leaf.flag was removed in Toy20"]
assert F.Standard(TOY, "Toy", 2005).problems(leaf(color="blue")) == ["Leaf.color 'blue' needs Toy10"]
assert F.Standard(TOY, "Toy", 1995).problems(leaf()) == ["Leaf needs Toy00"]
assert F.Standard(TOY, "Other", 2000).problems(leaf()) == ["Leaf is not Other"]
assert F.Standard(TOY, "Other", 2000).problems(Extra()) == []  # extensions are in every standard
assert F.Standard(TOY, "Other", 2000).problems(Pair()) == ["Pair is not Other"]  # by the language's base
three = Pair(left=leaf(), items=[leaf(), leaf(), leaf(color="blue")])
assert F.Standard(TOY, "Toy", 2005).check(three) == ["Pair with three items needs Toy30",
                                                      "items[2]: Leaf.color 'blue' needs Toy10"]
assert F.Standard(TOY, "Toy", 2030).check(three) == []
with raises(NotImplementedError, match="Toy15 has no parser"):
    toy2015.parse("")
with raises(NotImplementedError, match="Toy15 has no printer"):
    toy2015.print(leaf())
print("ok")

## SYN-09 · `children`, `walk`, `fold`, `same` and `copy`, with sharing and cycles

In [ ]:
x, y = leaf("x"), leaf("y")
tree = Pair(left=x, right=Pair(left=y, items=[x]), items=[y, x])
assert [n.value if isinstance(n, Leaf) else "pair" for n in F.walk(tree)] == ["pair", "x", "pair", "y"]
assert F.fold(tree, lambda n, rs: n.value if isinstance(n, Leaf) else "(" + " ".join(rs) + ")") == "(x (y x) y x)"
calls = []
F.fold(tree, lambda n, rs: calls.append(n))
assert len(calls) == 4  # shared syntax nodes once
cycle = Pair(items=[])
cycle.left = cycle
assert list(F.walk(cycle)) == [cycle]
with raises(ValueError, match="the tree contains a cycle"):
    F.fold(cycle, lambda n, rs: None)
assert F.same(tree, F.copy(tree)) and not F.same(tree, Pair(left=x))
assert not F.same(leaf("x"), leaf("y")) and not F.same(leaf("x"), Pair()) and not F.same(leaf(count=1), leaf(count=True))
assert not F.same(Pair(left=x, items=[x]), Pair(left=x, items=[y])) and not F.same(Pair(left=x), Pair(left=y))
assert F.same(None, None) and not F.same(x, None)
other = Pair(items=[])
other.left = other
assert F.same(cycle, other)
copied = F.copy(tree)
assert copied.left is copied.right.items[0] and copied.left is not x  # sharing is kept
loop = F.copy(cycle)
assert loop.left is loop and loop is not cycle
print("ok")

## SYN-10 · `Parents`: where syntax nodes are, and changing the tree in place

In [ ]:
a, b, c, d = leaf("a"), leaf("b"), leaf("c"), leaf("d")
inner = Pair(left=c)
tree = Pair(left=a, right=inner, items=[b, d])
parents = F.Parents(tree)
assert parents.parent(c) is inner and parents.parent(tree) is None and parents.location(d) == (tree, "items", 1)
assert list(parents.ancestors(c)) == [inner, tree] and parents.path(c) == "right.left" and parents.path(tree) == ""
assert parents.path(d) == "items[1]" and parents.location(leaf()) is None
e, f, g = leaf("e"), leaf("f"), Pair(left=leaf("h"))
parents.replace(b, [e, f])
assert [i.value for i in tree.items] == ["e", "f", "d"] and parents.location(d) == (tree, "items", 2)
parents.replace(a, g)
assert tree.left is g and parents.path(g.left) == "left.left"
parents.remove(e)
assert [i.value for i in tree.items] == ["f", "d"] and parents.location(d) == (tree, "items", 1)
parents.remove(inner)
assert tree.right is None and parents.parent(inner) is None
with raises(ValueError, match="the syntax node has no parent in this tree"):
    parents.replace(inner, leaf())
with raises(TypeError, match="Pair.left holds one syntax node, not a list"):
    parents.replace(g, [leaf()])
assert tree.left is g
print("ok")

## SYN-11 · `Visitor` and `Transformer` dispatch by kind

In [ ]:
class Values(F.Visitor):
    def __init__(self):
        self.found = []

    def visit_Leaf(self, node):
        self.found.append(node.value)


visitor = Values()
visitor.visit(Pair(left=leaf("a"), right=Pair(left=leaf("b")), items=[leaf("c")]))
assert visitor.found == ["a", "b", "c"]


class Rewrite(F.Transformer):
    def visit_Leaf(self, node):
        if node.value == "drop":
            return None
        if node.value == "twice":
            return [leaf("one"), leaf("two")]
        if node.value == "upper":
            return leaf("UPPER")
        return node


tree = Pair(left=leaf("upper"), right=leaf("drop"), items=[leaf("keep"), leaf("twice"), leaf("drop")])
assert Rewrite().visit(tree) is tree
assert tree.left.value == "UPPER" and tree.right is None and [i.value for i in tree.items] == ["keep", "one", "two"]
with raises(TypeError, match="Pair.left holds one syntax node, not a list"):
    Rewrite().visit(Pair(left=leaf("twice")))


class Rename(F.Transformer):
    def visit_Identifier(self, node):
        return S.Identifier(spelling=node.spelling.upper())


sum_ = S.BinaryExpression(left=S.IdExpression(name=S.Identifier(spelling="a")), operator="+",
                          right=S.IdExpression(name=S.Identifier(spelling="b")))
assert Rename().visit(sum_) is sum_ and (sum_.left.name.spelling, sum_.operator) == ("A", "+")
assert Rename().visit(S.ReturnStatement()).value is None
print("ok")